# Fraud Detection Analysis

This notebook focuses on analyzing transaction patterns, identifying anomalies, and developing fraud prevention strategies.

## Analysis Goals

1. **Transaction Pattern Analysis**
   - Time-based patterns
   - Amount distributions
   - Merchant category risks
   - Location-based patterns

2. **Anomaly Detection**
   - Unusual transaction patterns
   - Velocity checks
   - Amount anomalies
   - Location anomalies

3. **Customer Behavior Analysis**
   - Transaction frequency
   - Spending patterns
   - Risk indicators
   - Device usage patterns

4. **Fraud Prevention Strategies**
   - Risk scoring models
   - Early warning indicators
   - Prevention recommendations
   - Monitoring frameworks

## Expected Outcomes

1. Fraud pattern identification
2. Risk scoring recommendations
3. Prevention strategy framework
4. Monitoring system design

## Setup and Configuration

Import required libraries and configure the analysis environment.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report

# Import custom modules
import sys
sys.path.append(str(Path.cwd().parent))
from src.eda.fraud_analysis import FraudAnalyzer

# Visualization settings
plt.style.use('seaborn')
%matplotlib inline

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Set random seed for reproducibility
np.random.seed(42)

# Define data directory
DATA_DIR = Path('C:/Users/GIDI/Desktop/Folders/REPOSITORY/FinRisk-ai/data/raw')

## Data Loading and Preprocessing

Load and prepare transaction data for fraud analysis.

In [ ]:
# Load datasets
transactions = pd.read_csv(DATA_DIR / 'transaction_data.csv')
customer_data = pd.read_csv(DATA_DIR / 'customer_profiles.csv')

# Convert datetime columns
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

# Initialize FraudAnalyzer with our data
data = {
    'transactions': transactions,
    'customer_data': customer_data
}
analyzer = FraudAnalyzer(data)

# Calculate velocity metrics for different time windows
velocity_metrics = analyzer.calculate_velocity_metrics(['1H', '1D', '7D'])

print("Data Loading Complete!")
print("\nTransaction Dataset Shape:", transactions.shape)
print("Customer Data Shape:", customer_data.shape)

print("\nVelocity Metrics Summary:")
for window, metrics in velocity_metrics.items():
    print(f"\n{window} Window Metrics Shape:", metrics.shape)
    print(f"Sample of {window} Window Metrics:")
    print(metrics.head(3))

# Transaction Pattern Analysis

Analyze patterns in transaction data across different dimensions to identify potential fraud indicators.

In [ ]:
# Analyze transaction patterns
patterns = analyzer.analyze_patterns()

# Plot temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Temporal Transaction Patterns', fontsize=16)

# Hourly patterns
hourly = patterns['temporal_patterns']['hourly']
axes[0, 0].plot(hourly.index, hourly[('transaction_id', 'count')], marker='o')
axes[0, 0].set_title('Hourly Transaction Volume')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Number of Transactions')

# Daily patterns
daily = patterns['temporal_patterns']['daily']
axes[0, 1].plot(daily.index, daily[('transaction_id', 'count')], marker='o')
axes[0, 1].set_title('Daily Transaction Volume')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Number of Transactions')

# Plot merchant category patterns
merchant = patterns['merchant_patterns']
top_merchants = merchant.sort_values(('transaction_id', 'count'), ascending=False).head(10)
axes[1, 0].barh(range(len(top_merchants)), top_merchants[('transaction_id', 'count')])
axes[1, 0].set_yticks(range(len(top_merchants)))
axes[1, 0].set_yticklabels(top_merchants.index)
axes[1, 0].set_title('Top 10 Merchant Categories by Volume')

# Plot location patterns
location = patterns['location_patterns']
top_locations = location.sort_values(('fraud_flag', 'mean'), ascending=False).head(10)
axes[1, 1].barh(range(len(top_locations)), top_locations[('fraud_flag', 'mean')])
axes[1, 1].set_yticks(range(len(top_locations)))
axes[1, 1].set_yticklabels(top_locations.index)
axes[1, 1].set_title('Top 10 Locations by Fraud Rate')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nTransaction Pattern Summary:")
print("\nTop 5 Merchant Categories by Volume:")
print(merchant.sort_values(('transaction_id', 'count'), ascending=False).head()[[('transaction_id', 'count'), ('fraud_flag', 'mean')]])

print("\nTop 5 Locations by Fraud Rate:")
print(location.sort_values(('fraud_flag', 'mean'), ascending=False).head()[[('transaction_id', 'count'), ('fraud_flag', 'mean')]])

# Anomaly Detection Analysis

Detect anomalous transactions using multiple methods:
1. Z-score based detection for amount and velocity
2. Isolation Forest for multivariate detection

In [ ]:
# Detect anomalies using multiple methods
zscore_anomalies = analyzer.detect_anomalies(method='zscore', features=['amount', 'tx_count'])
iforest_anomalies = analyzer.detect_anomalies(method='isolation_forest', features=['amount', 'tx_count'])

# Plot anomaly detection results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Anomaly Detection Results', fontsize=16)

# Z-score anomalies
ax1.scatter(transactions['amount'], transactions['customer_id'], alpha=0.5, c='blue', label='Normal')
ax1.scatter(zscore_anomalies['amount'], zscore_anomalies['customer_id'], alpha=0.7, c='red', label='Anomalous')
ax1.set_title('Z-score Based Anomalies')
ax1.set_xlabel('Transaction Amount')
ax1.set_ylabel('Customer ID')
ax1.legend()

# Isolation Forest anomalies
ax2.scatter(transactions['amount'], transactions['customer_id'], alpha=0.5, c='blue', label='Normal')
ax2.scatter(iforest_anomalies['amount'], iforest_anomalies['customer_id'], alpha=0.7, c='red', label='Anomalous')
ax2.set_title('Isolation Forest Anomalies')
ax2.set_xlabel('Transaction Amount')
ax2.set_ylabel('Customer ID')
ax2.legend()

plt.tight_layout()
plt.show()

# Print anomaly detection summary
print("\nAnomaly Detection Summary:")
print(f"\nZ-score method detected {len(zscore_anomalies)} anomalous transactions ({len(zscore_anomalies)/len(transactions)*100:.2f}%)")
print(f"Isolation Forest detected {len(iforest_anomalies)} anomalous transactions ({len(iforest_anomalies)/len(transactions)*100:.2f}%)")

# Compare anomaly detection methods
zscore_fraud = zscore_anomalies['fraud_flag'].mean()
iforest_fraud = iforest_anomalies['fraud_flag'].mean()

print("\nFraud Rate in Anomalies:")
print(f"Z-score method: {zscore_fraud*100:.2f}%")
print(f"Isolation Forest: {iforest_fraud*100:.2f}%")

# Risk Segmentation Analysis

Identify high-risk segments across different dimensions:
1. Merchant categories
2. Locations
3. Time periods
4. Transaction amounts

In [ ]:
# Identify high-risk segments
risk_segments = analyzer.identify_high_risk_segments(min_transactions=50, risk_threshold=0.05)

# Plot high-risk segments
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('High-Risk Segment Analysis', fontsize=16)

# Merchant category risks
merchant_risks = risk_segments['merchant']
axes[0, 0].barh(range(len(merchant_risks)), merchant_risks['fraud_rate'])
axes[0, 0].set_yticks(range(len(merchant_risks)))
axes[0, 0].set_yticklabels(merchant_risks['merchant_category'])
axes[0, 0].set_title('High-Risk Merchant Categories')
axes[0, 0].set_xlabel('Fraud Rate')

# Location risks
location_risks = risk_segments['location']
axes[0, 1].barh(range(len(location_risks)), location_risks['fraud_rate'])
axes[0, 1].set_yticks(range(len(location_risks)))
axes[0, 1].set_yticklabels(location_risks['location'])
axes[0, 1].set_title('High-Risk Locations')
axes[0, 1].set_xlabel('Fraud Rate')

# Temporal risks
temporal_risks = risk_segments['temporal']
axes[1, 0].scatter(temporal_risks['hour'], temporal_risks['fraud_rate'])
axes[1, 0].set_title('Temporal Risk Patterns')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Fraud Rate')

# Amount range risks
amount_risks = risk_segments['amount']
axes[1, 1].plot(amount_risks.index, amount_risks['fraud_rate'], marker='o')
axes[1, 1].set_title('Transaction Amount Risk')
axes[1, 1].set_xlabel('Amount Range (Decile)')
axes[1, 1].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.show()

# Print risk segment summary
print("\nRisk Segment Summary:")
print("\nTop High-Risk Merchant Categories:")
print(merchant_risks[['transaction_count', 'fraud_rate', 'avg_amount']].head())

print("\nTop High-Risk Locations:")
print(location_risks[['transaction_count', 'fraud_rate', 'avg_amount']].head())

print("\nRiskiest Time Periods:")
print(temporal_risks.sort_values('fraud_rate', ascending=False)[['hour', 'is_weekend', 'transaction_count', 'fraud_rate']].head())

# Analysis Summary and Recommendations

Based on the analysis above, we can make the following observations and recommendations:

In [ ]:
# Generate comprehensive risk report
risk_report = analyzer.generate_risk_report()

# Print summary statistics
summary = risk_report['summary']
print("Overall Risk Analysis Summary:")
print("-" * 50)
print(f"Total Transactions: {summary['total_transactions'].values[0]:,}")
print(f"Total Fraud Cases: {summary['total_fraud'].values[0]:,}")
print(f"Overall Fraud Rate: {summary['fraud_rate'].values[0]*100:.2f}%")
print(f"Total Transaction Amount: ${summary['total_amount'].values[0]:,.2f}")
print(f"Average Transaction Amount: ${summary['avg_transaction'].values[0]:,.2f}")
print(f"Number of Unique Customers: {summary['unique_customers'].values[0]:,}")
print(f"Number of Unique Merchants: {summary['unique_merchants'].values[0]:,}")
print(f"Number of Unique Locations: {summary['unique_locations'].values[0]:,}")

print("\nKey Risk Indicators:")
print("-" * 50)

# Time-based risks
temporal_patterns = risk_report['patterns']['temporal_patterns']['hourly']
riskiest_hour = temporal_patterns[('fraud_flag', 'mean')].idxmax()
print(f"Riskiest Hour: {riskiest_hour:02d}:00 (Fraud Rate: {temporal_patterns[('fraud_flag', 'mean')][riskiest_hour]*100:.2f}%)")

# Merchant risks
merchant_patterns = risk_report['patterns']['merchant_patterns']
riskiest_merchant = merchant_patterns[('fraud_flag', 'mean')].idxmax()
print(f"Riskiest Merchant Category: {riskiest_merchant} (Fraud Rate: {merchant_patterns[('fraud_flag', 'mean')][riskiest_merchant]*100:.2f}%)")

# Location risks
location_patterns = risk_report['patterns']['location_patterns']
riskiest_location = location_patterns[('fraud_flag', 'mean')].idxmax()
print(f"Riskiest Location: {riskiest_location} (Fraud Rate: {location_patterns[('fraud_flag', 'mean')][riskiest_location]*100:.2f}%)")

print("\nRecommendations:")
print("-" * 50)
print("1. Enhanced Monitoring:")
high_risk_merchants = merchant_patterns[merchant_patterns[('fraud_flag', 'mean')] > 0.1].index.tolist()
print(f"   - Focus on {len(high_risk_merchants)} high-risk merchant categories")
print(f"   - Implement stricter controls during peak risk hours ({riskiest_hour:02d}:00)")

print("\n2. Risk Controls:")
print("   - Implement velocity checks for high-risk merchant categories")
print("   - Enhanced authentication for high-value transactions")
print("   - Geographic-based risk scoring")

print("\n3. Customer Segmentation:")
print("   - Develop risk profiles based on transaction patterns")
print("   - Implement customer-specific velocity limits")
print("   - Monitor customer behavior changes")